# **📘 Cleaning Employee Data for KLB Industries**

### **🎯 Objective**

You’re starting as a Data Engineer at KLB Industries (a defense contractor). Your first task is to clean and prepare an employee dataset for future use in a relational database.

---

### **📦 Dataset Overview**

🗂 File Name: `new_employee_data_messy.csv`

The dataset is a CSV file called **`new_employee_data_messy.csv`**. It was quickly thrown together by HR, so it might be quite a mess.

#### **It includes important fields such as:**

- 👤 **Name** – The name of an individual.  
- 🎓 **HighestDegree** – The highest academic degree the individual has obtained.  
- 🏢 **Division** – The division of KLB that the individual works in.  
- 🏢 **DivisionAddress** – The address of the KLB division where the individual works.  
- 🏠 **PersonalAddress** – The individual's personal address.  
- 📧 **Email** – The individual's email address. Clean every email to **`firstname_lastname_number@KLB.com`** (lowercase names, domain written exactly `KLB.com`), which is what the validator enforces. The raw domains are inconsistent (`klb.com`, `KLB.COM`, `klb..com`, `KLB..COM`).  
- 📞 **PhoneNumber** – The individual's phone number.  
- 💼 **Department** – The department in which the individual works.  
- 💼 **JobTitle** – The individual's job title.  
- 💼 **JobDescription** – The individual's job description.  
- 💰 **Salary** – The individual's annual salary.  
- 🎂 **Birthdate** – The individual's birthdate.  
- 🆔 **SocialSecurity** – The individual's Social Security Number (SSN).  

---

### **Your Current Task**

You need to **develop and apply data-cleaning techniques** to fix various issues in the dataset.

#### **🚧 Cleaning Tasks**
- 🔠 **Correct capitalization** and **remove extraneous special characters**.  
- ❓ **Fill missing values** where possible.  
- 🧼 **Validate and reformat fields** like SSN, email, and phone numbers.  
- 🔁 **Remove duplicate rows** (only if they are exact duplicates).  
- 🔗 **Ensure logical consistency** (e.g., job titles must align with departments).  

### **Final Step**

💾 **Export final version as** `Cleaned_data.csv` (exactly this name — the validator and Assignment 5 both look for `Cleaned_data.csv`).  

---

### 🚀 **Now, you're ready to transform messy employee data into a structured dataset!**


###  **🧰 Step 1: Read CSV file into pandas dataframe and inspect**

In [1]:
## Display in Notebook
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full column width

import numpy as np
import os
from collections import Counter
import re

### **📥 Step 2: Load the CSV File**

This section loads the dataset from a CSV file into a Pandas DataFrame.

1. **File Path**:
   - Update the `csv_file` variable with the correct path to the file on your local machine.
   - If you're unsure about the file's location, you can right-click the file and copy its full path.

2. **Understanding the Data**:
   - `df.info()`: Provides a summary of the data structure, including column names, data types, and the number of missing values.
   - `df.Name.head(10)`: Displays the first 10 names from the dataset to ensure the data loaded properly.

In [2]:
# Define the path to your CSV file
# The CSV lives in the SAME folder as this notebook, so a relative path works on any machine.

csv_file = "new_employee_data_messy.csv"

try:
    # Attempt to read the CSV file
    df = pd.read_csv(csv_file)
    print("✅ Data loaded successfully!")
    df.info()  # Display DataFrame information

except FileNotFoundError:
    print(f"❌ Error: The file '{csv_file}' was not found.")

except pd.errors.EmptyDataError:
    print(f"❌ Error: The file '{csv_file}' is empty.")

except pd.errors.ParserError:
    print(f"❌ Error: The file '{csv_file}' contains malformed data.")

except Exception as e:
    print(f"⚠️ An unexpected error occurred: {e}")
    

✅ Data loaded successfully!
<class 'pandas.DataFrame'>
RangeIndex: 15752 entries, 0 to 15751
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   EmployeeID             15667 non-null  float64
 1   Name                   15752 non-null  str    
 2   HighestDegree          15670 non-null  str    
 3   Division               15679 non-null  str    
 4   DivisionAddress        15698 non-null  str    
 5   DivisionPhone          15676 non-null  str    
 6   PersonalAddress        15678 non-null  str    
 7   Email                  15687 non-null  str    
 8   PhoneNumber            15676 non-null  str    
 9   Department             15690 non-null  str    
 10  DepartmentDescription  15692 non-null  str    
 11  JobTitle               15687 non-null  str    
 12  JobDescription         15662 non-null  str    
 13  Salary                 15687 non-null  float64
 14  Birthdate              15696 non-null

### **🔍 Step 3: Initial Data Inspection**

#### 👯‍♂️ Check for Duplicates and Drop Them


1. **Why Check for Duplicates?**
   - Duplicate rows can inflate metrics or bias results in analysis.
   - Removing duplicates ensures data integrity.

2. **Key Steps**:
   - `df.duplicated().sum()`: Counts how many rows are duplicates.
   - `df.drop_duplicates()`: Removes duplicate rows, keeping only the first occurrence.

This ensures the dataset is clean and contains only unique records.

In [3]:
# Count the number of duplicate rows
duplicate_count = df.duplicated().sum()

# Print the count of duplicates
print(f'📊 Total rows: {len(df)}')
print(f"🔁 Duplicate rows: {duplicate_count}")
df = df.drop_duplicates()
print(f'Number of unique rows: {len(df)}')

📊 Total rows: 15752
🔁 Duplicate rows: 1431
Number of unique rows: 14321


### **🧹 Step 4: Explore & Handle Missing Values**

We’ll look at:
- 🔍 Which columns have missing values?
- 🤔 How many values are missing?
- 🧠 Should we fill them or drop them?

1. **Why Standardize Missing Values?**
   - Different formats of missing values (`NaN`, `None`, empty strings) can cause confusion or errors during data processing.
   - Using `pd.NA` provides consistency across the dataset.

2. **Key Steps**:
   - `df.replace({np.nan: pd.NA})`: Converts all `NaN` values to `pd.NA`.
   - `df.isna().sum()`: Counts missing values for each column.
   - `df.isnull().any(axis=1).sum()`: Calculates how many rows have at least one missing value.

Use this information to decide how to handle missing values in the next steps (e.g., imputing, removing, or leaving as is). Run this cell and analyze your dataset!

In [4]:
# Identify and Standardize Missing Values

# Replace all NaN values with Pandas' NA representation
df = df.replace({np.nan: pd.NA})

# Count the number of missing values in each column
missing_values = df.isna().sum()

# Count the number of rows with at least one missing value
num_rows_with_missing = df.isnull().any(axis=1).sum()

# Display results
print(f" Number of Rows with Missing Values: {num_rows_with_missing}\n")

print(" Missing Values Count Per Column (Before Cleaning):")
print(missing_values)


 Number of Rows with Missing Values: 899

 Missing Values Count Per Column (Before Cleaning):
EmployeeID               80
Name                      0
HighestDegree            74
Division                 68
DivisionAddress          50
DivisionPhone            64
PersonalAddress          72
Email                    56
PhoneNumber              70
Department               55
DepartmentDescription    54
JobTitle                 62
JobDescription           81
Salary                   59
Birthdate                51
SocialSecurity           46
dtype: int64


### **🧼 Step 5: Data Cleaning and Feature Transformation**

To analyze the dataset effectively, we must ensure the data is clean and consistent. This step involves:
1. **Examining Each Feature**: Identify issues like inconsistent capitalization, special characters, or missing values.
2. **Applying Transformations**: Modify the data as needed to improve its quality.
3. **Splitting and Extracting**: Break complex fields (like `Name`) into meaningful components (e.g., First Name, Last Name).<br>
<br>


### **👤 Name Field Cleaning & Transformation**

- **Clean and Transform the `Name` Field**:
  - Fix capitalization and remove special characters.
  - Extract prefixes, suffixes, and degrees.
  - Split the name into components (e.g., Honorific, First Name, Last Name).<br>
<br>



In [5]:
#🧪 Inspect current state of the name column
df['Name'].head(50)

0                Steven Sean Johnson
1             Robert Matthew Schultz
2                Donna Brittany Ford
3     TERESA JENNIFER HERNANDEZ, THD
4            Christopher Perry Lopez
5             Maj Paul Carlos Guerra
6                 Martin David Brown
7           Catherine Rebecca Hurley
8           TONY KRISTOPHER MORRISON
9                 MICHAEL BARRY COOK
10        Genrl Kylie Donna Mitchell
11                   Martin John Cox
12               Kayla Amber Salazar
13                Robert Jeremy Wong
14           Margaret Karen Lee, ThD
15                 Justin Dean Brown
16                Lauren Mia Flowers
17              Autumn Joanna Coffey
18            Zachary William Miller
19            Tiffany Adriana Burton
20               James Richard Davis
21            KEITH DARREN RODRIGUEZ
22          Anthony Gregory Bell, BA
23           Edward Dalton Wilkerson
24              Joseph Timothy Smith
25              Kimberly Robin Boone
26              Tina Valerie Spencer
2

#### **🎩 Identify the honorifics in the data set**
We have to separate the honorifics from the first name. 
1. Split the name and look to see if the first element of the split is in a corpus of first names.
2. If not, it is an honorific.

#### **📥 Load First Name Corpus**
1. Got to https://www.ssa.gov/oact/babynames/limits.html
2. Click 'National Data'
3. It will download a compressed file of baby names from 1880's to now.
4. Extract this into a `data/first_names` folder **next to this notebook** (the code below creates `data/first_names` for you — just extract the txt files into it).
5. Note there will be about 145 txt files (`yob1880.txt` onward); the code below reads `yob1880`–`yob2022`

> ⚠️ **Note:** `names.zip` is bot-blocked for scripts — automated downloads (requests/curl) will fail. Download it via your browser from ssa.gov, or use the copy staged on Canvas.


In [8]:

# Folder where SSA files are extracted (relative to this notebook)
ssa_folder = "data/first_names"
os.makedirs(ssa_folder, exist_ok=True)  # creates the folder if it doesn't exist — extract the SSA txt files here

# Initialize an empty set for all names
ssa_names = set()

# Loop through all yobYYYY.txt files
for year in range(1880, 2023):  # From 1880 to 2022
    file_path = os.path.join(ssa_folder, f"yob{year}.txt")
    df2 = pd.read_csv(file_path, names=['name', 'gender', 'count'])
    
    # Add names to the set
    ssa_names.update(df2['name'].str.lower())

# Test if a name exists in the full SSA dataset
test_name = "Yesenia"
if test_name.lower() in ssa_names:
    print(f"{test_name} is a valid first name.")
else:
    print(f"{test_name} is NOT in the dataset.")

# Test if a name exists in the full SSA dataset
test_name = "Jackie"
if test_name.lower() in ssa_names:
    print(f"{test_name} is a valid first name.")
else:
    print(f"{test_name} is NOT in the dataset.")


test_name = "XXX"
if test_name.lower() in ssa_names:
    print(f"✅ {test_name} is a valid first name.")
else:
    print(f"❌ {test_name} is NOT in the dataset.")

Yesenia is a valid first name.
Jackie is a valid first name.
❌ XXX is NOT in the dataset.


In [9]:

# 🧼 Function to clean and process the name
def process_name_first_token(name):
    # Remove special characters, periods, commas
    cleaned_name = re.sub(r'[^a-zA-Z\s]', '', name)  # Keep only letters and spaces
    cleaned_name = cleaned_name.strip()  # Remove leading/trailing whitespace
    
    # Split the name into parts
    name_parts = cleaned_name.split()
    
    # Return the first token or an empty string if the name is empty
    return name_parts[0] if name_parts else ''

# Initialize a set to store unique unmatched tokens
unmatched_tokens = set()

# Process each name in the DataFrame
for full_name in df['Name']:  # Note: Ensure column name matches your DataFrame
    first_token = process_name_first_token(full_name)  # Extract the first token
    # Normalize capitalization
    normalized_token = first_token.lower()
    # Check against the corpus and add to unmatched_tokens if not found
    if normalized_token and normalized_token not in ssa_names:
        unmatched_tokens.add(normalized_token)

# Output the unique unmatched tokens
print("Unique unmatched tokens (possible titles/honorifics):")
for token in sorted(unmatched_tokens):
    print(token)


Unique unmatched tokens (possible titles/honorifics):
adm
capt
cmdr
col
cpl
dr
genrl
hon
lt
maj
prof
prv
sgt


#### **🎓 Create Honorific and Degree Lists**
#### At this point we have two lists. 
1. **honorifics** ~ titles and appearing before the first name
2. **degrees_list** ~ degree acronymes occurring at the end <br>

Now, we want to split each name into its various parts, and create new dataframe columns **'Honorific'**, **'First Name'**,**'Middle Name'**, **'Last Name'**, **'Degree'**

In [10]:
# Convert unmatched_tokens to a sorted list
honorifics_list = sorted(unmatched_tokens)  

# Capitalize each honorific correctly
honorifics_list = [honorific.capitalize() for honorific in honorifics_list]

degree_list = ['PhD', 'MD', 'JD', 'DO', 'DDS', 'DVM', 'EdD', 'LLD', 'ScD', 'ThD',
                   'MBA', 'MS', 'MA', 'MSc', 'MEd', 'MFA', 'MLA', 'LLM',
                   'BS', 'BA', 'BSc', 'BFA', 'LLB', 'AB'
                  ]

# Print the formatted list
print(honorifics_list)
print(degree_list)

['Adm', 'Capt', 'Cmdr', 'Col', 'Cpl', 'Dr', 'Genrl', 'Hon', 'Lt', 'Maj', 'Prof', 'Prv', 'Sgt']
['PhD', 'MD', 'JD', 'DO', 'DDS', 'DVM', 'EdD', 'LLD', 'ScD', 'ThD', 'MBA', 'MS', 'MA', 'MSc', 'MEd', 'MFA', 'MLA', 'LLM', 'BS', 'BA', 'BSc', 'BFA', 'LLB', 'AB']


#### **🔎 Cleaning and Splitting the `Name` Field**

The `Name` column contains full names, including prefixes (e.g., "Dr."), suffixes (e.g., "Jr."), and degree titles (e.g., "PhD"). To analyze the data effectively, we need to:
1. **Fix Issues**:
   - Remove unnecessary special characters.
   - Ensure consistent capitalization.
   - Handle missing or empty names gracefully.
2. **Extract Components**:
   - Split the name into meaningful parts:
     - **Honorific**: e.g., "Dr.", "Prof."
     - **First Name**: The first part of the name.
     - **Middle Name(s)**: All names between the first and last name.
     - **Last Name**: The final part of the name.
     - **Degree**: e.g., "PhD", "MBA".
3. **Output Example**:
   - Input: `"Dr. John A. Smith, PhD"`
   - Output:
     ```
     Honorific: Dr
     First Name: John
     Middle Name: A
     Last Name: Smith
     Degree: PHD
     ```
4. **Regular Expressions (Regex)**:
   - We'll use regex to identify and extract patterns like honorifics and suffixes. If you're new to regex, check out:
     - [Regex Tutorial #1](https://www.youtube.com/watch?v=EzeeypMKx7o)
     - [Regex Tutorial #2](https://www.youtube.com/watch?v=j6A28L6Tmxw)


#### **🏷️ Handle the Prefixes and Suffixes**

To clean and process the `Name` column, we need to:
1. **Split Prefixes and Suffixes**:
   - Extract honorifics (e.g., "Dr.", "Prof.") into a separate column.
   - Extract degree suffixes (e.g., "PhD", "MBA").
2. **Clean the Remaining Name**:
   - Normalize capitalization.
   - Remove special characters.
   - Split the remaining name into `First Name`, `Middle Name`, and `Last Name`.
3. **Update Columns**:
   - Add columns for `Honorific`, `First Name`, `Middle Name`, `Last Name`, and `Degree`.

This ensures the `Name` column is fully processed into clean, structured components.

In [ ]:
import pandas as pd
import re

# Step 1: Define a function to clean and process names
def process_name(full_name):
    # Step 1a: Remove commas and normalize spaces
    name = re.sub(r'[,\s]+', ' ', full_name.strip())  # Replace commas and extra spaces with a single space

    # Patterns for matching honorifics, name suffixes, and degree suffixes
    honorifics_pattern = r'\b(?:' + '|'.join(re.escape(h) for h in honorifics_list) + r')\b'
    degree_suffix_pattern = r'\b(?:' + '|'.join(re.escape(d) for d in degree_list) + r')\b'

    # FIX: Ensure degrees are fully removed BEFORE further processing
    prefix_remove = r'\b(?:Mr|Mrs|Ms|Miss)\.?\b'  # Remove non-honorific prefixes

    # Step 2: Extract and remove degree suffix (ensure full removal)
    degree_match = re.search(degree_suffix_pattern, name, flags=re.IGNORECASE)
    degree = degree_match.group(0).upper() if degree_match else ''
    
    if degree:
        name = re.sub(r'\b' + re.escape(degree) + r'\b', '', name, flags=re.IGNORECASE).strip()  # FORCE REMOVE DEGREE

    # Step 3: Remove non-honorific prefixes (e.g., "Mr.", "Mrs.")
    name = re.sub(prefix_remove, '', name, flags=re.IGNORECASE).strip()

    # Step 4: Extract and remove honorifics from the name
    honorific_match = re.search(honorifics_pattern, name, flags=re.IGNORECASE)
    honorific = honorific_match.group(0).capitalize() if honorific_match else ''
    name = re.sub(honorifics_pattern, '', name, flags=re.IGNORECASE).strip()

    # Step 5: Ensure spaces are normalized after removals
    name = re.sub(r'\s+', ' ', name).strip()  # Remove extra spaces again

    # Step 6: Split the cleaned name into parts
    parts = name.split()

    # Step 7: Assign first, middle, and last names
    first_name = parts[0].capitalize() if len(parts) > 0 else ''
    last_name = parts[-1].capitalize() if len(parts) > 1 else ''
    middle_name = ' '.join(p.capitalize() for p in parts[1:-1]) if len(parts) > 2 else ''

    return pd.Series([honorific, first_name, middle_name, last_name, degree])


# Step 2: Apply the function to the 'Name' column in the DataFrame
df[['Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree']] = df['Name'].apply(process_name)

In [ ]:
# Step 3: Display the processed DataFrame in Jupyter Notebook
from IPython.display import display
display(df[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', "HighestDegree"]].head(10))

#### **Observations**
1. Special characters have been removed, and capitalization has been normalized across the dataset.
2. In some cases, the degree extracted from the `Name` column (`Degree`) does not match the value in the `HighestDegree` column.
   - For example, a name might contain "PhD" while `HighestDegree` lists "Master's."
3. To maintain consistency:
   - If the `Degree` column is not empty and its value differs from `HighestDegree`, we will replace the value in `HighestDegree` with the one from `Degree`.
4. Once the `HighestDegree` column is updated, the `Degree` column will be dropped to avoid redundancy.
5. Some degrees do not match the honorifics. For example, doctors will have some type of advanced degree not GED.
6. If a person has one of these honorifics \[Adm, Capt, Cmdr, Col, Dr, Genrl, Hon, Lt, Maj, Prof,\] and one of these highest degrees 
\[AB, HS, CTE, GED\], replace highest degree with 'Bachelors+'

#### **Let's look more closely**

In [ ]:
# Set display options for better alignment
pd.set_option('display.colheader_justify', 'center')  # Center column headers
pd.set_option('display.max_columns', None)  # Ensure all columns are shown
pd.set_option('display.width', 1000)  # Expand width for better readability

print(df[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].head(10).to_string(index=False))


In [ ]:
import pandas as pd
import numpy as np

# Convert pd.NA back to np.nan for consistency
df = df.replace({pd.NA: np.nan})

# Define honorifics that should not have a low-level degree
advanced_honorifics = {'Adm', 'Capt', 'Cmdr', 'Col', 'Dr', 'Genrl', 'Hon', 'Lt', 'Maj', 'Prof'}

# Define degrees that should be replaced if paired with an advanced honorific
low_degrees = {'AB', 'HS', 'CTE', 'GED'}

# Define function to update HighestDegree based on Degree
def update_highest_degree(row):
    """
    Updates 'HighestDegree' with 'Degree' if 'Degree' is not empty 
    and 'HighestDegree' is either empty or different.
    """
    if pd.notna(row["Degree"]) and row["Degree"].strip():  # Ensure 'Degree' is not missing or empty
        if pd.isna(row["HighestDegree"]) or row["Degree"].strip() != row["HighestDegree"].strip():
            return row["Degree"].strip()  # Replace 'HighestDegree' with 'Degree'
    return row["HighestDegree"]  # Retain the original 'HighestDegree'

# Apply function to update 'HighestDegree'
df["HighestDegree"] = df.apply(update_highest_degree, axis=1)

# Step 2: Replace low degrees with 'Bachelors+' if honorific is in the advanced list
df.loc[df['Honorific'].isin(advanced_honorifics) & df['HighestDegree'].isin(low_degrees), 'HighestDegree'] = 'Bachelors+'

# # Step 3: Drop the redundant 'Degree' column (only if it exists)
# if 'Degree' in df.columns:
#     df.drop(columns=['Degree'], inplace=True)

# Print formatted table
print(df[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].head(100).to_string(index=False))



In [ ]:
## Look at honorifics and degrees
# Display rows where 'Degree' column is not empty (not NaN)
df_with_honorific = df[df['Honorific'].notna() & (df['Honorific'] != '')]

# Display the first 50 matching rows
print(df_with_honorific[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].head(50))

# Display the last 50 matching rows
print(df_with_honorific[['Name', 'Honorific', 'First Name', 'Middle Name', 'Last Name', 'Degree', 'HighestDegree']].tail(50))


In [ ]:

df.info()

#### **🏢 Division**

The `Division` column contains the names of divisions where employees work. However, these names may:
1. Contain special characters.
2. Have inconsistent capitalization.
3. Include invalid or unrecognized division names.
4. Have missing values.

To address these issues, we will:
1. Retain missing values as they are.
2. Remove special characters from division names.
3. Normalize capitalization to title case.
4. Verify that each division name matches a predefined list of valid divisions.
5. Missing or unrecognized divisions will be set to `NaN`.



In [ ]:
# Step 1: Explore the Division column
# - View the first 20 rows of the Division and DivisionAddress columns
print("First 20 rows of the 'Division' column:")
print(df.Division.head(20))

In [ ]:
# Step 2: Define a list of valid divisions
# - This list contains all acceptable division names

# 'Corporate' is NOT valid: those rows are set to missing here and get their division back from
# DivisionAddress (456 South Ave = South Division) in the fill step further down.
Divisions = ['North Division', 'South Division', 'East Division', 'West Division', 'Central Division']

# Step 3: Define a function to clean and validate the Division column
def clean_division(division):
    # Retain NaN values
    if pd.isna(division):  # Check if the value is NaN
        return division  # Return NaN as it is

    # Step 3.1: Convert to string (if not already)
    division = str(division)

    # Step 3.2: Remove special characters
    division = re.sub(r'[^a-zA-Z0-9\s,]', '', division)

    # Step 3.3: Normalize capitalization (convert to title case)
    division = division.title()

    # Step 3.4: Validate against the predefined list
    if division not in Divisions:
        print(f"Unrecognized division: {division}")  # Debug output for unrecognized divisions
        return pd.NA  # Mark as missing if not valid
    
    return division  # Return the cleaned division name

    

> **The validator accepts only the 5 geographic divisions.** The few `Corporate` rows will print as *Unrecognized division* below and become missing; that is expected. Their `DivisionAddress` is the South Division address, so the **Fill Missing `Division`** step below restores them as `South Division`.


In [ ]:
# Step 4: Apply the cleaning function to the 'Division' column
df["Division"] = df["Division"].apply(clean_division)

# Step 5: Check for missing values in the Division column
missing_division = df["Division"].isna().sum()
print(f"\nMissing values in 'Division': {missing_division}")

# Step 6: Display the first 10 rows of the CleanedDivision column
print("\nFirst 10 rows of 'Division':")
print(df.Division.head(10))

#### **🏠 Divison Addresses**

The `DivisionAddress` column contains the addresses associated with each division. These addresses may:
1. Contain unnecessary special characters.
2. Have inconsistent capitalization.
3. Include improperly formatted state abbreviations.

To address these issues, we will:
1. Retain missing values as they are.
2. Remove special characters (except spaces).
3. Normalize capitalization (e.g., "123 MAIN st" → "123 Main St").
4. Ensure state abbreviations are correctly capitalized (e.g., "tx" → "TX").

We will use a predefined list of valid state abbreviations to ensure accuracy.


In [ ]:
df.DivisionAddress.head(20)

##### **📍 Reference — Correct addresses and state abbreviations given below**

In [ ]:
# Step 1: Define the correct addresses for each division
Division_Addresses = [
    {
        'division': 'North Division',
        'address': '123 North St, Springfield, IL 62701',
        'city': 'Springfield',
        'state': 'IL',
        'zip_code': '62701',
        'phone': '(217)555-1234'
    },
    {
        'division': 'South Division',
        'address': '456 South Ave, Austin, TX 73301',
        'city': 'Austin',
        'state': 'TX',
        'zip_code': '73301',
        'phone': '(512)555-6789'
    },
    {
        'division': 'East Division',
        'address': '789 East Blvd, Albany, NY 12201',
        'city': 'Albany',
        'state': 'NY',
        'zip_code': '12201',
        'phone': '(518)555-1234'
    },
    {
        'division': 'West Division',
        'address': '101 West Dr, Sacramento, CA 94203',
        'city': 'Sacramento',
        'state': 'CA',
        'zip_code': '94203',
        'phone': '(916)555-5678'
    },
    {
        'division': 'Central Division',
        'address': '202 Main St, Columbus, OH 43085',
        'city': 'Columbus',
        'state': 'OH',
        'zip_code': '43085',
        'phone': '(916)555-6789'
    }
]


# List of all valid state abbreviations
STATE_ABBREVIATIONS = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", 
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", 
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", 
    "WI", "WY"
]

# Step 2: Create a regex pattern to match state abbreviations
# - Matches any valid two-letter state abbreviation (case-insensitive)
STATE_PATTERN = r'\b(' + '|'.join(STATE_ABBREVIATIONS) + r')\b'
# Display the regex pattern (for debugging purposes)
print(STATE_PATTERN)

In [ ]:
# Step 3: Define a function to clean and normalize addresses
def clean_address(address):
    # Ensure the input is a string and handle NaN values
    if pd.isna(address):  # Check if the value is NaN
        return address  # Return NaN as it is
    address = str(address)  # Convert to string if not already

    # Remove special characters (except spaces)
    address = re.sub(r'[^a-zA-Z0-9\s,]', '', address)

    # Remove commas
    address = address.replace(',', '')

    # Normalize capitalization (title case for the address)
    address = address.title()  # note that this turns TX in Tx, so we must correct

    # Find and capitalize the state abbreviation
    def capitalize_state(match):
        return match.group(0).upper()

    # Apply the regex pattern to capitalize state abbreviations
    # Note that STATE_PATTERN is matched against address, the match is sent to function capitalize_state, 
    # where is gets converted to all caps, then the substituted into the string address.
    address = re.sub(STATE_PATTERN, capitalize_state, address, flags=re.IGNORECASE)

    return address

In [ ]:
# Apply the function to clean the Address
df["DivisionAddress"] = df["DivisionAddress"].apply(clean_address)

# Check for missing values in CleanedDivisionAddress
missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in DivisionAddress: {missing_division_address}")

df.DivisionAddress.head(15)

In [ ]:
# Check for missing values in Division
# - This counts the number of rows where Division is NaN (missing).

missing_division = df["Division"].isna().sum()
print(f"Missing values in Division: {missing_division}")

# Check for missing values in DivisionAddress
# - This counts the number of rows where DivisionAddress is NaN (missing).

missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in DivisionAddress: {missing_division_address}")

# Identify rows where both Division and DivisionAddress are missing
# - This creates a filtered DataFrame with rows where both columns are NaN.
missing_rows = df[df["Division"].isna() & df["DivisionAddress"].isna()]

# Display the number of rows with missing values in both columns
print(f"Number rows with missing values in both 'Division' and 'DivisionAddress': {len(missing_rows)}")
print("Rows with missing values in both 'Division' and 'DivisionAddress':")

print(missing_rows.index.tolist()) 

In [ ]:
# Try to fill in missing values where both are missing

# Define mapping of states to divisions
state_to_division = {
    'IL': 'North Division',
    'TX': 'South Division',
    'NY': 'East Division',
    'CA': 'West Division',   
    'OH': 'Central Division'   
}

# Define mapping of divisions to addresses
division_address_map = {entry['division']: entry['address'] for entry in Division_Addresses}

# Function to update missing Division and DivisionAddress
def fix_missing_division(row):
    if pd.isna(row['Division']) and pd.isna(row['DivisionAddress']):
        # Extract state from PersonalAddress
        state_match = next((state for state in state_to_division if state in str(row['PersonalAddress'])), None)
        if state_match:
            division = state_to_division[state_match]
            division_address = division_address_map.get(division, None)
            row['Division'] = division
            row['DivisionAddress'] = division_address
    return row

# Apply the function to only the rows where both Division and DivisionAddress are missing
df.loc[df['Division'].isna() & df['DivisionAddress'].isna(), :] = df.loc[
    df['Division'].isna() & df['DivisionAddress'].isna()].apply(fix_missing_division, axis=1)

indices = [9, 126, 196]  # Replace with actual index values

# Print the Division and Division Address for the specified indices
print(df.loc[indices, ['Division', 'DivisionAddress']])


#### **🧩 Fill Missing `DivisionAddress` (When `Division` Is Present)**

To handle rows with a missing `DivisionAddress`, we will:
1. Use the `Division` column to identify the division.
2. Fill in the missing `DivisionAddress` using a predefined mapping of divisions to their correct addresses (`division_to_address`).
3. Leave the `DivisionAddress` as is for rows where both the division and address are missing.

This step ensures that every division has a corresponding address wherever possible.

In [ ]:
# Step 1: Create a mapping of divisions to their correct addresses
# - This uses the predefined Division_Addresses list to create a dictionary.
division_to_address = {entry['division']: entry['address'] for entry in Division_Addresses}

# Step 2: Define a function to fill missing division addresses
def fill_missing_division_address(row):
    
    # Step 2.1: Check if 'DivisionAddress' is missing
    if pd.isna(row['DivisionAddress']):
        # Step 2.2: Use 'Division' to look up the correct address
        return division_to_address.get(row['Division'], row['DivisionAddress'])
    
    # Step 2.3: Return the original address if it's not missing
    return row['DivisionAddress']


In [ ]:
# Step 3: Apply the function to the DataFrame
# - This updates the 'DivisionAddress' column with filled values where possible.
df['DivisionAddress'] = df.apply(fill_missing_division_address, axis=1)

# Step 4: Check for remaining missing values in the 'DivisionAddress' column
missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in 'DivisionAddress' after filling: {missing_division_address}")

# Step 5: Identify rows where 'Division' is present but 'DivisionAddress' is missing
rows_with_missing_address = df[df['DivisionAddress'].isna() & df['Division'].notna()]

# Display up to 5 rows showing 'Division' and 'DivisionAddress'
if not rows_with_missing_address.empty:
    print("\nRows with Division but missing DivisionAddress (showing up to 5):")
    print(rows_with_missing_address[['Division', 'DivisionAddress']].head(5).to_string(index=False))
    rows_with_missing_address
else:
    print("\nNo rows found where Division is present but DivisionAddress is missing.")


#### **🔄 Fill Missing `Division` (When `DivisionAddress` Is Present)**

In cases where the `Division` is missing but the `DivisionAddress` is available, we will:
1. Use the `DivisionAddress` column to identify the corresponding division.
2. Fill in the `Division` column using a predefined mapping of addresses to divisions (`address_to_division`).
3. Leave the `Division` as `NaN` if no valid address is found.

This step ensures that divisions are correctly filled based on their associated addresses.

In [ ]:
# Step 1: Normalize and create a mapping of addresses to their corresponding divisions
address_to_division = {
    entry['address'].replace(",", "").strip().lower(): entry['division']
    for entry in Division_Addresses
}

# Step 2: Define a function to fill missing divisions based on the address
def fill_missing_division(row):
    # Step 2.1: If 'Division' is already present, return it as is
    if not pd.isna(row['Division']):
        return row['Division']

    # Step 2.2: If 'DivisionAddress' is missing, return NaN
    if pd.isna(row['DivisionAddress']):
        return pd.NA

    # Step 2.3: Normalize 'DivisionAddress' for lookup (strip spaces, remove commas, convert to lowercase)
    address = row['DivisionAddress'].replace(",", "").strip().lower()

    # Step 2.4: Use 'DivisionAddress' to look up the corresponding division
    if address in address_to_division:
        return address_to_division[address]

    # Step 2.5: Log unmatched addresses for debugging
    unmatched_addresses.add(address)

    # Step 2.6: Return NaN if no matching division is found
    return pd.NA

# # Step 3: Initialize a set to track unmatched addresses
# unmatched_addresses = set()

# # Step 4: Apply the function to the DataFrame
# df['Division'] = df.apply(fill_missing_division, axis=1)

# # Step 5: Debug unmatched addresses
# print(f"Number of unmatched addresses: {len(unmatched_addresses)}")
# if unmatched_addresses:
#     print("Sample unmatched addresses:", list(unmatched_addresses)[:10])  # Print first 10 for review


In [ ]:

unmatched_addresses = set()  # used by fill_missing_division to log addresses it cannot match

# Step 3: Apply the function to the DataFrame
# - This updates the 'CleanedDivision' column with filled values where possible.
df['Division'] = df.apply(fill_missing_division, axis=1)

# Step 4: Check for remaining missing values in the CleanedDivision column
missing_division = df["Division"].isna().sum()
missing_division_address = df["DivisionAddress"].isna().sum()
print(f"Missing values in 'Division' after filling: {missing_division}")
print(f"Missing values in 'DivisionAddress' after filling: {missing_division_address}")

# Print formatted table
print(df[['Division', 'DivisionAddress']].head(10).to_string(index=False))


## **🧱 For the Student to Complete**
Break the Division address up into 4 parts.
1. `DivStreetAddress` "456 South Ave"
2. `DivCity` "Austin"
3. `DivState` "TX"
4. `DivZip` "73301"

Check your work carefully. Add these columns to the dataframe.

### **🏡 Personal Address**

The `PersonalAddress` column contains the addresses of individual employees. These addresses may:
1. Contain unnecessary special characters.
2. Have inconsistent capitalization.
3. Include formatting inconsistencies.

To clean the `PersonalAddress` column, we will:
1. Retain missing values as they are.
2. Remove special characters (except spaces).
3. Normalize capitalization to title case.

#### **🧽 Get Rid of Special Characters and Normalize Capitalization**

We will use the `clean_address` function to process the `PersonalAddress` column, ensuring all addresses are formatted consistently.

#### **🧪 Validate That Personal Address Is in the Same State as the Division the Person Works For**

In this step, we will:
1. Compare the `PersonalAddress` column with the `DivisionAddress` column to ensure the personal address is in the same state as the division address.
2. Validate each row by:
   - Extracting the state abbreviation from both addresses.
   - Comparing the extracted state abbreviations.
3. Mark each row as:
   - ✅ **Valid**: If the state matches.
   - ❌ **Invalid**: If the state does not match.
   - ⚠️ **Missing**: If one or both addresses are missing.

This ensures consistency between personal and division addresses.

####  **🔄 If Personal Address Is Missing, Replace with Division Address**

In cases where the `PersonalAddress` is missing, we will:
1. Replace it with the corresponding `DivisionAddress`.
2. Ensure that each row has a valid address wherever possible.

This step ensures that no personal address is left missing if a division address is available.

#### **🧱 Break the `PersonalAddress` into 4 Parts**

1. `PerStreetAddress` "101 West Dr"
2. `PerCity` "Sacramento"
3. `PerState` "CA"
4. `PerZip` "94203"

Check your work carefully. Add these columns to the dataframe.

### **📧 Email**

Should be in form, firstname_lastname_number\@KLB.com. If not, please repair.  
#### **🔍 Observations**

1. Capitalization not constant, need to make lower case
2. Sometimes there are honorifics, these should not be there
3. Sometimes the middle name is included, sometimes not.

➡️ Since we know the format required "firstname_lastname_number\@KLB.com", we will simply reconstruct all emails. We need to get the unique number from each. 

Modifications: 
1. From each email, extract the number.
2. Then reconstruct the email as shown above
3. Replace all emails with your newly constructed email.

### **☎️ Phone Number**

1. Should be in the form: `(929)614-7016`, where `929` is the area code  
2. If there is an extension, remove it  
3. There is no need for a country code since all employees are in the US  
4. If missing, fill in with the corresponding **division phone number**  
5. When finished, validate that there are **no missing phone numbers**  
6. If the phone is missing, replace it with the division phone number (if available)

### **🏢 Department**
1. Should be one of the following:
   - `'Human Resources'`, `'Operations'`, `'Engineering'`, `'Administrative'`, `'Executive'`, `'Management'`  
2. If missing, try to infer from the **job title** or **job description**  
   - Use reference file: `job_descriptions.csv`

> **Use `'Human Resources'`, not `'HR'`.** The validator takes the valid departments straight from `job_descriptions.csv`.


### **💼 Job Title**

1. Inspect and clean the job title  
2. Make sure the job title matches one of the standard titles  
   - Refer to the `job_descriptions.csv` file  
3. If the title is missing, try to infer it from the **job description**  
4. Ensure the job title is consistent with the assigned **department**

### **📝 Job Description**

1. Inspect and clean the job description  
2. Ensure it matches one of the **standard descriptions**  
   - Refer to the `job_descriptions.csv` file for validation

### **✅ Perform Validation Steps**

1. Check if the employee’s `JobDescription` exists in the CSV  
2. Check if the employee’s `JobTitle` exists in the CSV  
3. Ensure `JobTitle` and `JobDescription` are consistent with each other  
4. Parse `JobDescription` for division references and validate against the employee’s `Division`  
5. Report all findings

#### **🧼 Please Finish the Following Steps**

1. Inspect the data and make **observations** (in a markdown cell) about what needs to be done  
2. Remove any **extraneous characters**  
3. Normalize **capitalization** across all fields  
4. Perform any other required **cleaning tasks**  
5. **Save** the cleaned data  
6. **Validate** the cleaned data to ensure integrity

### **🔎 Job Title / Description Validity Check**

1. For `MISSING_JOB_DESCRIPTIONS`, use the CSV file to fill in  
2. For `MISSING_JOB_TITLES`, use the CSV file to fill in  
3. For `INCONSISTENT_JOB_TITLE_DESCRIPTION`, assume the **job title** is correct and fix the **job description**  
4. For `INCONSISTENT_DIVISIONS`, assume the **Division** entry is correct and fix the **job description**  
5. Then, rerun the validation step and regenerate the **DETAILED VALIDATION REPORT**

### **🆔 EmployeeID**

1. Make sure each ID is **cleaned**  
2. Add a column to the dataframe called `'InvalidID'`  
3. Check for **duplicates**  
   - If found, replace all with a new unique ID **within that employee's department range** (see `department_ID_ranges` below)  
   - Add `"duplicate"` to the `'InvalidID'` column  
4. Check for appropriate **ID range**  
   - If an ID is out of range, add `"range"` to the `'InvalidID'` column  
5. Check for **missing values**  
   - If missing, generate a new unique ID **within that employee's department range** (see `department_ID_ranges` below)  
   - Add `"missing"` to the `'InvalidID'` column  
6. If the ID is valid, add `"valid"` to the `'InvalidID'` column

You can use the two dictionaries below if needed:

**Define department ID ranges** <br>
department_ID_ranges = {<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Human Resources': (100000, 199999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Operations': (200000, 299999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Engineering': (300000, 399999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Administrative': (400000, 499999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Executive': (500000, 599999),<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Management': (500000, 599999)<br>
}<br>

**Define JobTitle keyword to department mapping** (check the keywords **in this order and stop at the first match** — e.g. *Chief Operations Officer* is Executive, not Operations)<br>
jobtitle_to_department = {<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Chief': 'Executive',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Division Manager': 'Management',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Human Resources': 'Human Resources',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'HR': 'Human Resources',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Operations': 'Operations',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Engineer': 'Engineering',<br>
    &nbsp;&nbsp;&nbsp;&nbsp;'Secretary': 'Administrative'<br>
}


### **💰 Salary**

1. Check whether the salary is within the specified range for the corresponding **job title**  
   - Refer to the `salary_ranges.csv` file  
2. If the salary is **out of range** or **missing**, set it to the **average salary** for that job title

### **🎂 Birthdate**

Ensure that the birthdate is in the proper format: `MM/DD/YYYY`  
📌 Example: `08/01/1960` (two-digit month and day, four-digit year)

**~51 rows have missing Birthdate. You MUST impute them (document your strategy) — leaving them null will exceed the 50-error budget.** Dates appear in exactly two formats: M/D/YYYY and YYYY.MM.DD.


### **🔐 Social Security Number**

Ensure that the Social Security Number is in the proper format: `XXX-XX-XXXX`  
📌 Example: `777-35-0022`

#### **🛠️ Note**
If any SSN is **missing**, generate a **unique ID** in the format `XXX-XX-XXXX`  
➡️ Ensure no duplicates exist in the generated values.

## **💾 Save DataFrame to CSV**

Save the cleaned DataFrame as **`Cleaned_data.csv`** in this folder, e.g. `df.to_csv("Cleaned_data.csv", index=False)`.

The validator requires these 24 columns, spelled exactly like this (extra working columns are reported but not counted as errors):

`EmployeeID`, `Name`, `HighestDegree`, `Division`, `DivisionAddress`, `DivisionPhone`, `DivStreetAddress`, `DivCity`, `DivState`, `DivZip`, `PersonalAddress`, `PerStreetAddress`, `PerCity`, `PerState`, `PerZip`, `Email`, `PhoneNumber`, `Department`, `DepartmentDescription`, `JobTitle`, `JobDescription`, `Salary`, `Birthdate`, `SocialSecurity`

`Name` must be the cleaned full name (title case, no honorific or degree).

📌 We will use this cleaned dataset in the next assignments.